# 04 · SKU & Section Classification

```
inventory-analysis/
├── notebooks/  04_sku_collection_classification.ipynb
├── src/        classification.py
└── outputs/    FABRIC_ANALYSIS_COMPLETE.xlsx
```

## 0 · Dependencies

In [ ]:
!pip install pandas numpy scipy statsmodels tqdm plotly openpyxl -q

## 1 · Imports

In [37]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
import classification as cl
print('imports ok')

imports ok


## 2 · Load data

In [38]:
combined_df = pd.read_parquet(PROJECT_ROOT / 'outputs' / 'combined_df.parquet')
print(f'shape : {combined_df.shape}')
print(f'range : {combined_df["Date"].min().date()} → {combined_df["Date"].max().date()}')

shape : (200000, 33)
range : 2019-01-01 → 2025-12-30


## 3 · Run classification

In [39]:
sku_df, section_df = cl.ultimate_fabric_analysis(
    combined_df,
    analysis_date=None,
    save_excel=True,
    output_file=str(PROJECT_ROOT / 'outputs' / 'FABRIC_ANALYSIS_COMPLETE.xlsx'),
)

# Normalise section revenue column name (pipeline uses 'Recent_Revenue')
if section_df is not None and 'Recent_Revenue' in section_df.columns and 'Recent_Revenue_12M' not in section_df.columns:
    section_df = section_df.rename(columns={'Recent_Revenue': 'Recent_Revenue_12M',
                                             'Recent_Qty': 'Recent_Quantity_12M'})
print(f'SKUs: {len(sku_df):,}  |  Sections: {len(section_df) if section_df is not None else 0}')


── Classification Pipeline ───────────────────────────────────
  analysis date : 2025-12-30
  unique SKUs   : 1,943
  transactions  : 200,000
  profit data   : yes
  stock data    : yes

── Pre-computing section seasonality ─────────────────────────


STL: 100%|█████████████████████████████████████| 62/62 [00:00<00:00, 184.81it/s]



── Stage 1 · Scalar metrics (vectorised) ─────────────────────
  1,943 SKUs

── Stage 2 · Monthly trend signals ──────────────────────────


SKUs: 100%|████████████████████████████████| 1943/1943 [00:06<00:00, 295.57it/s]


  1,943 SKUs

── Stage 3 · Lost-sales estimation ──────────────────────────
  2 stockout SKUs processed

── Stage 4 · Status assignment (vectorised) ─────────────────
  1,943 SKUs classified

── Stage 5 · Section analysis ───────────────────────────────


Sections: 100%|████████████████████████████████| 62/62 [00:00<00:00, 156.51it/s]


  62 sections

── Saving → D:\Projects\InventoryDeepDive\outputs\FABRIC_ANALYSIS_COMPLETE.xlsx
  saved  ·  D:\Projects\InventoryDeepDive\outputs\FABRIC_ANALYSIS_COMPLETE.xlsx
SKUs: 1,943  |  Sections: 62


## 4 · Dashboards

### 4.1 · Portfolio health matrix

**What you're looking at:** Every bubble is one status group (e.g. *Growing*, *Declining*, *True Stockout*). Position on **X** = average recent margin of that group. Position on **Y** = what share of your total revenue that group represents. Bubble size = number of SKUs in the group.

**How to read it:** A bubble in the top-left (high revenue share, negative margin) means you're generating a lot of revenue but losing money on it — the most dangerous position. A bubble in the top-right (high revenue share, strong margin) is your core profit engine. Bubbles near the bottom don't matter much regardless of margin because they barely contribute to revenue.

**Action:** Any large bubble left of the red break-even line needs immediate pricing or cost review.

In [41]:
grp = sku_df.groupby('Status').agg(
    SKU_Count         = ('Status',              'count'),
    Avg_Margin        = ('Recent_Avg_Margin_%',  'mean'),
    Revenue_Share     = ('Recent_Revenue',        'sum'),
    Avg_Lost_Revenue  = ('Estimated_Lost_Revenue','mean'),
    Priority          = ('Priority',              'min'),
).reset_index()
grp['Revenue_Share_%'] = grp['Revenue_Share'] / grp['Revenue_Share'].sum() * 100
grp['Avg_Margin'] = grp['Avg_Margin'].clip(-30, 60)

COLOR_MAP = {
    'True Stockout':'#d62728','Loss-Making':'#d62728',
    'Potential Stockout (High Value)':'#ff7f0e','Low Stock (High Margin)':'#ff7f0e',
    'Stockout':'#ffbb78','Low Stock':'#ffbb78',
    'Declining (Low Margin)':'#e377c2','Declining':'#f7b6d2',
    'Slow Mover':'#8c564b','Dead':'#7f7f7f','Inactive':'#c7c7c7',
    'Growing (High Margin)':'#2ca02c','Growing':'#98df8a',
    'Stable (Profitable)':'#1f77b4','Stable':'#aec7e8',
    'Seasonal':'#9467bd','New':'#17becf','Volatile':'#bcbd22',
}

fig = px.scatter(
    grp,
    x='Avg_Margin', y='Revenue_Share_%',
    size='SKU_Count', size_max=60,
    color='Status', color_discrete_map=COLOR_MAP,
    text='Status',
    hover_data={'SKU_Count': True, 'Avg_Margin': ':.1f',
                'Revenue_Share_%': ':.1f', 'Avg_Lost_Revenue': ':,.0f'},
    title='Portfolio Health Matrix  ·  Status groups by margin vs revenue share',
    labels={'Avg_Margin': 'Avg Recent Margin %', 'Revenue_Share_%': '% of Total Revenue'},
)
fig.add_vline(x=0,  line_dash='dash', line_color='red',   annotation_text='Break-even')
fig.add_vline(x=25, line_dash='dot',  line_color='green', annotation_text='25% target')
fig.update_traces(textposition='top center', textfont_size=9)
fig.update_layout(height=600, showlegend=False,
                  plot_bgcolor='black', paper_bgcolor='black')
fig.show()

### 4.2 · ABC revenue waterfall

**What you're looking at:** SKUs ranked from highest to lowest revenue (X-axis = cumulative % of SKUs). The curve shows how much of total revenue is captured as you include more SKUs.

**How to read it:** The further the curve bows toward the top-left corner, the more concentrated your revenue is. A perfectly equal portfolio would follow the grey diagonal — every SKU contributes the same. The vertical dashed lines show what your top 20% and top 50% of SKUs actually generate.

**Action:** If your top 20% generate more than 80% of revenue, you have high concentration risk — losing one key SKU or having it stock out has an outsized impact.

In [42]:
rev_sorted = sku_df[['SKU', 'Status', 'Recent_Revenue']].copy()
rev_sorted = rev_sorted.sort_values('Recent_Revenue', ascending=False).reset_index(drop=True)
rev_sorted['Cum_Pct'] = rev_sorted['Recent_Revenue'].cumsum() / rev_sorted['Recent_Revenue'].sum() * 100
rev_sorted['SKU_Pct']  = (rev_sorted.index + 1) / len(rev_sorted) * 100

# Sample to 500 points for rendering speed
sample = rev_sorted.iloc[::max(1, len(rev_sorted)//500)].copy()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=sample['SKU_Pct'], y=sample['Cum_Pct'],
    mode='lines', line=dict(color='#1f77b4', width=2.5),
    fill='tozeroy', fillcolor='rgba(31,119,180,0.12)',
    name='Cumulative Revenue',
))
fig.add_trace(go.Scatter(
    x=[0, 100], y=[0, 100],
    mode='lines', line=dict(color='grey', dash='dot', width=1),
    name='Perfect equality', showlegend=True,
))
for skup, label, color in [(20, 'Top 20%', '#d62728'), (50, 'Top 50%', '#ff7f0e')]:
    rev_at = float(rev_sorted.iloc[int(len(rev_sorted) * skup / 100)]['Cum_Pct'])
    fig.add_vline(x=skup, line_dash='dash', line_color=color,
                  annotation_text=f'{label} → {rev_at:.0f}% revenue',
                  annotation_position='top right')

fig.update_layout(
    title='ABC Revenue Concentration  ·  Cumulative revenue vs SKU rank',
    xaxis_title='% of SKUs (ranked by revenue)',
    yaxis_title='Cumulative % of Revenue',
    height=500, plot_bgcolor='black', paper_bgcolor='black',
)
fig.show()

### 4.3 · Margin distribution by revenue trend

**What you're looking at:** The full distribution of recent margin % for each trend group. The wide part of each violin shows where most SKUs cluster. The box inside shows median and interquartile range.

**How to read it:** Ideally, the *Growing* violin should sit clearly above the *Declining* violin. If *Growing* SKUs have a similar or lower margin distribution than *Declining* ones, it means you're buying volume by discounting — growing revenue without growing profit.

**Action:** Growing SKUs below the 25% target line (green dot) are worth investigating — are they growing because of price cuts?

In [44]:
plot = sku_df[sku_df['Revenue_Trend'].isin(['Growing','Stable','Declining']) &
              sku_df['Recent_Avg_Margin_%'].between(-40, 80)].copy()

fig = px.violin(
    plot, x='Revenue_Trend', y='Recent_Avg_Margin_%',
    color='Revenue_Trend',
    color_discrete_map={'Growing':'#2ca02c','Stable':'#1f77b4','Declining':'#d62728'},
    box=True, points=False,
    title='Margin Distribution by Revenue Trend',
    labels={'Revenue_Trend': '', 'Recent_Avg_Margin_%': 'Recent Margin %'},
    category_orders={'Revenue_Trend': ['Growing','Stable','Declining']},
)
fig.add_hline(y=0,  line_dash='dash', line_color='red',   annotation_text='Break-even')
fig.add_hline(y=25, line_dash='dot',  line_color='green', annotation_text='25% target')
fig.update_layout(height=500, showlegend=False,
                  plot_bgcolor='black', paper_bgcolor='black')
fig.show()

### 4.4 · Section risk radar

**What you're looking at:** Each polygon is one section. Five axes, each scored 0–10 where **10 = most risk**:

- **Margin risk** — how far below the 50% ceiling (inverted: high score = low margin)
- **Stock risk** — % of SKUs currently out of stock
- **Trend risk** — % of SKUs with a declining revenue trend
- **Concentration risk** — how much revenue comes from the top 20% of SKUs
- **Activity risk** — how many days since the last sale (recency)

**How to read it:** A section with a large filled area on all five axes is high-risk across every dimension. A spike on only one axis tells you exactly where the problem is.

**Action:** Sections with large polygons and spikes on Stock + Trend axes simultaneously are in structural decline, not just a stock issue.

In [46]:
if section_df is not None and len(section_df) >= 3:
    top8 = section_df.nlargest(8, 'Recent_Revenue_12M').copy()

    def norm(s, lo, hi, invert=False):
        v = (s.clip(lo, hi) - lo) / (hi - lo) * 10
        return 10 - v if invert else v

    top8['r_margin']  = norm(top8['Recent_Avg_Margin_%'], -20, 50, invert=True)
    top8['r_stock']   = norm(top8['Stock_Out_Rate_%'],      0, 60)
    top8['r_trend']   = norm(top8['Pct_Declining'],         0, 80)
    top8['r_conc']    = norm(top8['Top20_Contribution_%'],  0, 100)
    top8['r_activity']= norm(top8['Days_Since_Last_Sale'],  0, 180)

    dims = ['Margin risk','Stock risk','Trend risk','Concentration risk','Activity risk']
    cols = ['r_margin','r_stock','r_trend','r_conc','r_activity']

    fig = go.Figure()
    colors = px.colors.qualitative.Set2
    for i, row in top8.iterrows():
        vals = [row[c] for c in cols]
        vals += vals[:1]
        fig.add_trace(go.Scatterpolar(
            r=vals, theta=dims + [dims[0]],
            name=str(row['Section']),
            line=dict(width=2, color=colors[i % len(colors)]),
            fill='toself', opacity=0.25,
        ))
    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, 10])),
        title='Section Risk Radar  ·  Top 8 Sections by Revenue  (higher = more risk)',
        height=580, paper_bgcolor='black',
    )
    fig.show()
else:
    print('section_df not available')

### 4.5 · Stock-out opportunity map

**What you're looking at:** Each bubble is a stockout SKU. **X** = how many days it has been out of stock. **Y** = the SAR per day it was generating before it ran out (pre-stockout daily revenue rate). Bubble size = estimated total revenue lost during the stockout period. Red = confirmed True Stockout (was selling consistently, then stopped). Orange = Potential.

**How to read it:** Top-right corner = worst situation: out of stock for a long time AND was a high-velocity seller. Bottom-left = low urgency: barely sold before and hasn't been out long.

**Action:** Prioritise reorders by working top-right to bottom-left. The bubble size confirms the financial cost of delay.

In [48]:
so = sku_df[
    (sku_df['Is_Stockout'] | sku_df['Is_True_Stockout']) &
    (sku_df['Estimated_Lost_Revenue'] > 0)
].copy()

if len(so):
    so['Daily_Rate'] = np.where(
        so['Days_Out_of_Stock'] > 0,
        so['Estimated_Lost_Revenue'] / so['Days_Out_of_Stock'].clip(lower=1),
        0
    )
    fig = px.scatter(
        so.nlargest(60, 'Estimated_Lost_Revenue'),
        x='Days_Out_of_Stock', y='Daily_Rate',
        size='Estimated_Lost_Revenue', size_max=40,
        color='Is_True_Stockout',
        color_discrete_map={True: '#d62728', False: '#ff7f0e'},
        hover_data=['SKU', 'Section', 'Estimated_Lost_Revenue',
                    'Recent_Avg_Margin_%', 'Key_Reason'],
        labels={
            'Days_Out_of_Stock': 'Days Out of Stock',
            'Daily_Rate': 'Daily Revenue Rate (SAR)',
            'Is_True_Stockout': 'True Stockout',
        },
        title='Stock-out Opportunity Map  ·  Top 60 by lost revenue  (size = est. lost SAR)',
    )
    fig.update_layout(height=560, plot_bgcolor='black', paper_bgcolor='black')
    fig.show()
    print(f'Total est. lost revenue : {so["Estimated_Lost_Revenue"].sum():,.0f} SAR')
    print(f'Total est. lost profit  : {so["Estimated_Lost_Profit"].sum():,.0f} SAR')
else:
    print('No stockouts detected.')

Total est. lost revenue : 6,854 SAR
Total est. lost profit  : 2,940 SAR


### 4.6 · Section performance quadrant

**What you're looking at:** A BCG-style matrix. **X** = recent margin %, **Y** = YoY revenue change %. Bubble size = 12-month revenue. The grey lines divide the chart into four quadrants.

| Quadrant | Name | Meaning |
|---|---|---|
| Top-right | ⭐ Stars | High growth + high margin — invest and protect |
| Bottom-right | 🐄 Cash Cows | Low growth + high margin — harvest profit |
| Top-left | ❓ Question Marks | High growth + low margin — decide: fix margin or exit |
| Bottom-left | 🐕 Dogs | Low growth + low margin — review for discontinuation |

**Action:** Large bubbles in Dogs or Question Marks represent the biggest decisions — they carry meaningful revenue but either aren't growing or aren't profitable.

In [50]:
if section_df is not None:
    q = section_df[
        section_df['Revenue_YoY_Change_%'].notna() &
        section_df['Recent_Avg_Margin_%'].between(-30, 70)
    ].nlargest(40, 'Recent_Revenue_12M').copy()

    q['yoy'] = q['Revenue_YoY_Change_%'].clip(-60, 80)

    fig = px.scatter(
        q,
        x='Recent_Avg_Margin_%', y='yoy',
        size='Recent_Revenue_12M', size_max=50,
        color='Status',
        hover_data=['Section', 'Key_Reason', 'Total_SKUs',
                    'Pct_Growing', 'Stock_Out_Rate_%'],
        text='Section',
        title='Section Performance Quadrant  ·  Top 40 by Revenue',
        labels={
            'Recent_Avg_Margin_%': 'Recent Margin %',
            'yoy': 'YoY Revenue Change %',
        },
    )
    fig.add_hline(y=0,  line_dash='dash', line_color='grey')
    fig.add_vline(x=20, line_dash='dash', line_color='grey')

    for label, x, y, anchor in [
        ('⭐ Stars',        55,  55, 'right'),
        ('🐄 Cash Cows',    55, -40, 'right'),
        ('❓ Question Marks', 2,  55, 'left'),
        ('🐕 Dogs',          2, -40, 'left'),
    ]:
        fig.add_annotation(x=x, y=y, text=label, showarrow=False,
                           font=dict(size=11, color='grey'), xanchor=anchor)

    fig.update_traces(textposition='top center', textfont_size=8)
    fig.update_layout(height=600, plot_bgcolor='rgba(248,248,248,1)',
                      paper_bgcolor='black')
    fig.show()

## 5 · Summary tables

### Critical SKUs

In [51]:
from IPython.display import display

cols = ['SKU','Section','Status','Key_Reason','Current_Stock',
        'Recent_Revenue','Recent_Avg_Margin_%','Days_Since_Last',
        'Estimated_Lost_Revenue']
present = [c for c in cols if c in sku_df.columns]
fmt = {c: v for c, v in {
    'Recent_Revenue': '{:,.0f}',
    'Recent_Avg_Margin_%': '{:.1f}%',
    'Estimated_Lost_Revenue': '{:,.0f}'
}.items() if c in present}

styled = sku_df[sku_df['Priority'] <= 5][present].head(25).style.format(fmt)
if 'Recent_Avg_Margin_%' in present:
    styled = styled.background_gradient(
        subset=['Recent_Avg_Margin_%'], cmap='RdYlGn', vmin=-20, vmax=40)
display(styled)

,SKU,Section,Status,Key_Reason,Current_Stock,Recent_Revenue,Recent_Avg_Margin_%,Days_Since_Last,Estimated_Lost_Revenue
1043,55750260011,5575,True Stockout,Sudden stop after consistent sales,0,"49,345",44.0%,41,"5,827"
1443,52340040025,5234,True Stockout,Sudden stop after consistent sales,0,"1,651",43.5%,47,"1,026"
192,50490040014,5049,Low Stock (High Margin),1 units + 38.8% margin,1,"435,824",38.8%,3,0
985,31910170011,3191,Low Stock (High Margin),7 units + 48.1% margin | Revenue UP / Qty DOWN → price increase or mix shift,7,"318,364",48.1%,7,0
344,52340050012,5234,Low Stock (High Margin),1 units + 46.2% margin | Revenue stable / Qty Growing,1,"275,637",46.2%,0,0
97,40990100016,4099,Low Stock (High Margin),2 units + 51.7% margin,2,"255,278",51.7%,8,0
789,45010270014,4501,Low Stock (High Margin),1 units + 47.7% margin,1,"245,874",47.7%,16,0
1183,50920160028,5092,Low Stock (High Margin),1 units + 48.9% margin | Revenue stable / Qty Declining,1,"244,820",48.9%,0,0
748,50490060024,5049,Low Stock (High Margin),2 units + 45.0% margin,2,"236,226",45.0%,23,0
821,45390090025,4539,Low Stock (High Margin),5 units + 48.4% margin,5,"147,126",48.4%,8,0


### Section summary

In [52]:
from IPython.display import display

if section_df is not None:
    cols = ['Section','Status','Key_Reason','Total_SKUs','Recent_Revenue_12M',
            'Recent_Avg_Margin_%','Pct_Growing','Pct_Declining',
            'Stock_Out_Rate_%','Inventory_Turnover_Ratio','Is_Top_Heavy']
    present = [c for c in cols if c in section_df.columns]
    fmt = {c: v for c, v in {
        'Recent_Revenue_12M': '{:,.0f}',
        'Recent_Avg_Margin_%': '{:.1f}%',
        'Pct_Growing': '{:.1f}%',
        'Pct_Declining': '{:.1f}%',
        'Stock_Out_Rate_%': '{:.1f}%',
        'Inventory_Turnover_Ratio': '{:.1f}x',
    }.items() if c in present}

    styled = section_df[present].head(20).style.format(fmt)
    if 'Recent_Avg_Margin_%' in present:
        styled = styled.background_gradient(
            subset=['Recent_Avg_Margin_%'], cmap='RdYlGn', vmin=-20, vmax=40)
    display(styled)

,Section,Status,Key_Reason,Total_SKUs,Recent_Revenue_12M,Recent_Avg_Margin_%,Pct_Growing,Pct_Declining,Stock_Out_Rate_%,Inventory_Turnover_Ratio,Is_Top_Heavy
34,4663,Growing (High Margin),"Revenue up, 47.5% margin",53,"8,474,663",47.5%,41.5%,34.0%,0.0%,579.3x,False
33,3267,Declining,Revenue trending down,53,"7,338,799",45.2%,30.2%,39.6%,0.0%,544.5x,False
23,4351,Growing (High Margin),"Revenue up, 48.3% margin",50,"7,320,462",48.3%,34.0%,44.0%,0.0%,616.3x,False
32,3131,Seasonal,Clear seasonal pattern,51,"7,218,090",45.2%,35.3%,43.1%,0.0%,548.4x,False
27,5575,Declining,Revenue trending down,54,"6,839,536",47.7%,31.5%,55.6%,1.9%,587.7x,False
39,5482,Growing (High Margin),"Revenue up, 45.6% margin",41,"6,798,052",45.6%,46.3%,41.5%,0.0%,700.8x,False
31,3276,Growing (High Margin),"Revenue up, 47.3% margin",44,"6,058,807",47.3%,31.8%,43.2%,0.0%,594.9x,False
11,5912,Growing (High Margin),"Revenue up, 47.4% margin",41,"5,953,747",47.4%,36.6%,36.6%,0.0%,691.1x,False
4,4579,Declining,Revenue trending down,45,"5,756,100",44.2%,35.6%,44.4%,0.0%,541.6x,False
20,5358,Declining,Revenue trending down,51,"5,605,320",46.0%,23.5%,54.9%,0.0%,383.8x,False


In [1]:
combined_df.columns

NameError: name 'combined_df' is not defined